In [17]:
import torch
from transformers import SwinForImageClassification, SwinConfig
from torchinfo import summary

model_name = "microsoft/swin-tiny-patch4-window7-224"
config = SwinConfig.from_pretrained(model_name)  # 先加载配置
model = SwinForImageClassification.from_pretrained(
    model_name, 
    config=config,
    ignore_mismatched_sizes=True  # 忽略分类头维度不匹配（如需仅看结构，可加）
)

model.eval()
summary(model, input_size=(1, 3, 224, 224))

Loading weights:   0%|          | 0/233 [00:00<?, ?it/s]

Layer (type:depth-idx)                                            Output Shape              Param #
SwinForImageClassification                                        [1, 1000]                 --
├─SwinModel: 1-1                                                  [1, 768]                  --
│    └─SwinEmbeddings: 2-1                                        [1, 3136, 96]             --
│    │    └─SwinPatchEmbeddings: 3-1                              [1, 3136, 96]             4,704
│    │    └─LayerNorm: 3-2                                        [1, 3136, 96]             192
│    │    └─Dropout: 3-3                                          [1, 3136, 96]             --
│    └─SwinEncoder: 2-2                                           [1, 49, 768]              --
│    │    └─ModuleList: 3-4                                       --                        27,512,922
│    └─LayerNorm: 2-3                                             [1, 49, 768]              1,536
│    └─AdaptiveAvgPool1d: 2-4 

- `Swin-Tiny`的模型参数:
```json
SwinConfig {
  "architectures": [
    "SwinForImageClassification"
  ],
  "attention_probs_dropout_prob": 0.0,
  "depths": [
    2,
    2,
    6,
    2
  ],
  "drop_path_rate": 0.1,
  "dtype": "float32",
  "embed_dim": 96,
  "encoder_stride": 32,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.0,
  "hidden_size": 768,
  "id2label": {
    "0": "tench, Tinca tinca",
    "1": "goldfish, Carassius auratus",
    "2": "great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias",
    "3": "tiger shark, Galeocerdo cuvieri",
    "4": "hammerhead, hammerhead shark",
    "5": "electric ray, crampfish, numbfish, torpedo",
  },
  "image_size": 224,
  "initializer_range": 0.02,
  "label2id": {
    "Afghan hound, Afghan": 160,
    "African chameleon, Chamaeleo chamaeleon": 47,
    "African crocodile, Nile crocodile, Crocodylus niloticus": 49,
    "African elephant, Loxodonta africana": 386,
    "African grey, African gray, Psittacus erithacus": 87,
  },
  "layer_norm_eps": 1e-05,
  "mlp_ratio": 4.0,
  "model_type": "swin",
  "num_channels": 3,
  "num_heads": [
    3,
    6,
    12,
    24
  ],
  "num_layers": 4,
  "out_features": [
    "stage4"
  ],
  "out_indices": [
    4
  ],
  "patch_size": 4,
  "path_norm": true,
  "qkv_bias": true,
  "stage_names": [
    "stem",
    "stage1",
    "stage2",
    "stage3",
    "stage4"
  ],
  "transformers_version": "5.1.0",
  "use_absolute_embeddings": false,
  "window_size": 7
}
```

In [40]:
device = torch.device("mps")
# 每阶段，每个Swin层的shift_size
for i, stage in enumerate(model.swin.encoder.layers):
    for j, layer in enumerate(stage.blocks):
        # SwinModel -> SwinEncoder -> SwinStage -> SwinLayer
        # model.swin.encoder.layers[i].blocks[j].get_attn_mask()
        attn = layer.get_attn_mask(56, 56, dtype=torch.float32, device=device)
        out = "None" if attn is None else attn.shape
        print(f"stage{i}.layer{j}, shift_size: {layer.shift_size} attn: {out}")

stage0.layer0, shift_size: 0 attn: None
stage0.layer1, shift_size: 3 attn: torch.Size([64, 49, 49])
stage1.layer0, shift_size: 0 attn: None
stage1.layer1, shift_size: 3 attn: torch.Size([64, 49, 49])
stage2.layer0, shift_size: 0 attn: None
stage2.layer1, shift_size: 3 attn: torch.Size([64, 49, 49])
stage2.layer2, shift_size: 0 attn: None
stage2.layer3, shift_size: 3 attn: torch.Size([64, 49, 49])
stage2.layer4, shift_size: 0 attn: None
stage2.layer5, shift_size: 3 attn: torch.Size([64, 49, 49])
stage3.layer0, shift_size: 0 attn: None
stage3.layer1, shift_size: 0 attn: None
